In [2]:
#install necessary libraries
!pip install transformers torch

In [3]:
#import necessary libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [4]:
model_name = "microsoft/DialoGPT-small"  # lighter + faster than medium

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Model loaded successfully!")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully!


In [9]:
# Demonstrate how tokenization works
sample_text = "Hello, how are you?"

# Encode text to token IDs
encoded = tokenizer.encode(sample_text, return_tensors="pt")
print(f"Original text: {sample_text}")
print(f"Token IDs: {encoded}")
print(f"Individual tokens: {tokenizer.tokenize(sample_text)}")

# Decode back to text
decoded = tokenizer.decode(encoded[0], skip_special_tokens=True)
print(f"Decoded text: {decoded}")

Original text: Hello, how are you?
Token IDs: tensor([[15496,    11,   703,   389,   345,    30]])
Individual tokens: ['Hello', ',', 'Ġhow', 'Ġare', 'Ġyou', '?']
Decoded text: Hello, how are you?


In [10]:
def generate_response(user_input, chat_history_ids=None, max_length=1000):
    """
    Generate a response from the chatbot for a given user input.

    Parameters:
    - user_input: The user's message (string)
    - chat_history_ids: Previous conversation history (tensor)
    - max_length: Maximum length of generated response

    Returns:
    - response: The chatbot's response (string)
    - chat_history_ids: Updated conversation history (tensor)
    """

    # Step 5a: Encode the user input and add end-of-sequence token
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,  # Add EOS token
        return_tensors="pt"                 # Return PyTorch tensor
    )

    # Step 5b: Append new input to chat history (if exists)
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Step 5c: Generate response using the model
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=max_length,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,       # Avoid repeating 3-word phrases
        do_sample=True,                # Enable sampling for diverse responses
        top_k=50,                      # Consider top 50 tokens
        top_p=0.95,                    # Nucleus sampling threshold
        temperature=0.75               # Control randomness (lower = focused)
    )

    # Step 5d: Decode only the new response (not the full history)
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return response, chat_history_ids

# Test the function
test_response, test_history = generate_response("Hello, how are you?")
print(f"User: Hello, how are you?")
print(f"Bot: {test_response}")

User: Hello, how are you?
Bot: Hello, how are your friend


In [11]:
# Test with different inputs to see model capability
test_inputs = [
    "What is Artificial Intelligence?",
    "Tell me a joke",
    "What is Python programming?",
    "Who are you?"
]

print("=" * 60)
print("TESTING INDIVIDUAL RESPONSES")
print("=" * 60)

for user_input in test_inputs:
    response, _ = generate_response(user_input)
    print(f"\nUser: {user_input}")
    print(f"Bot:  {response}")
    print("-" * 40)


TESTING INDIVIDUAL RESPONSES

User: What is Artificial Intelligence?
Bot:  What is AI?
----------------------------------------

User: Tell me a joke
Bot:  What is Aleppo's meaning?
----------------------------------------

User: What is Python programming?
Bot:  Python. Python is programming. Python.
----------------------------------------

User: Who are you?
Bot:  I know I have
----------------------------------------


In [12]:
def chatbot():
    """
    Main chatbot function that handles continuous conversation.
    The chatbot maintains conversation history for context-aware responses.
    """

    # Initialize conversation history
    chat_history_ids = None
    step_count = 0  # Track conversation turns

    # Display welcome message
    print("=" * 60)
    print("       AI CHATBOT - Powered by DialoGPT")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant.")
    print("         How can I help you today?")
    print("         (Type 'exit' or 'quit' to end)")
    print("=" * 60)

    # Main conversation loop
    while True:
        # Step 7a: Get user input
        user_input = input("\nYou: ").strip()

        # Step 7b: Check for empty input
        if not user_input:
            print("Chatbot: Please type something. I'm here to help!")
            continue

        # Step 7c: Check exit condition
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print("\nChatbot: Goodbye! It was nice talking to you.")
            print("         Have a great day! 👋")
            print("=" * 60)
            break

        # Step 7d: Generate response
        try:
            response, chat_history_ids = generate_response(
                user_input,
                chat_history_ids
            )
            step_count += 1

            # Step 7e: Reset history if it gets too long (prevent memory issues)
            if step_count > 5:
                chat_history_ids = None
                step_count = 0

            # Step 7f: Display response
            print(f"Chatbot: {response}")

        except Exception as e:
            print(f"Chatbot: I'm sorry, I encountered an error. Let me reset.")
            print(f"         Error: {str(e)}")
            chat_history_ids = None
            step_count = 0

# Run the chatbot
chatbot()

       AI CHATBOT - Powered by DialoGPT
Chatbot: Hello! I am your AI assistant.
         How can I help you today?
         (Type 'exit' or 'quit' to end)

You: Hey! How are you?
Chatbot: Hey fellow Canadian!

You: Can't tell you about python.
Chatbot: What language is that?

You: Goodbye! It was nice talking to you.
Chatbot: Sorry. Just got back.

You: exit

Chatbot: Goodbye! It was nice talking to you.
         Have a great day! 👋


In [13]:
from datetime import datetime

def enhanced_chatbot():
    """
    Enhanced chatbot with conversation logging and additional features.
    """

    # Initialize variables
    chat_history_ids = None
    step_count = 0
    conversation_log = []  # Store conversation for display

    # Welcome message
    print("\n" + "=" * 60)
    print("    🤖 ENHANCED AI CHATBOT - DialoGPT Medium")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant powered by")
    print("         Microsoft DialoGPT.")
    print("         How can I help you today?")
    print("\n📌 Commands:")
    print("   • Type 'exit' or 'quit' to end conversation")
    print("   • Type 'history' to view conversation log")
    print("   • Type 'reset' to clear conversation context")
    print("=" * 60)

    while True:
        # Get user input
        user_input = input("\n👤 You: ").strip()

        # Handle empty input
        if not user_input:
            print("🤖 Chatbot: I didn't catch that. Could you please type something?")
            continue

        # Handle exit
        if user_input.lower() in ['exit', 'quit', 'bye', 'goodbye']:
            print("\n🤖 Chatbot: Goodbye! Thanks for chatting with me! 👋")

            # Show conversation summary
            print(f"\n📊 Conversation Summary:")
            print(f"   Total exchanges: {len(conversation_log)}")
            print("=" * 60)
            break

        # Handle history command
        if user_input.lower() == 'history':
            print("\n📜 Conversation History:")
            print("-" * 40)
            if conversation_log:
                for entry in conversation_log:
                    print(f"   👤 You: {entry['user']}")
                    print(f"   🤖 Bot: {entry['bot']}")
                    print(f"   ⏰ Time: {entry['time']}")
                    print("-" * 40)
            else:
                print("   No conversation history yet.")
            continue

        # Handle reset command
        if user_input.lower() == 'reset':
            chat_history_ids = None
            step_count = 0
            print("🤖 Chatbot: Conversation context has been reset!")
            continue

        # Generate response
        try:
            response, chat_history_ids = generate_response(
                user_input,
                chat_history_ids
            )
            step_count += 1

            # Log the conversation
            conversation_log.append({
                'user': user_input,
                'bot': response,
                'time': datetime.now().strftime("%H:%M:%S")
            })

            # Reset history periodically to prevent context overflow
            if step_count > 5:
                chat_history_ids = None
                step_count = 0

            # Display response
            print(f"🤖 Chatbot: {response}")

        except Exception as e:
            print(f"🤖 Chatbot: Sorry, let me try again. (Resetting context)")
            chat_history_ids = None
            step_count = 0

# Run the enhanced chatbot
enhanced_chatbot()


    🤖 ENHANCED AI CHATBOT - DialoGPT Medium
Chatbot: Hello! I am your AI assistant powered by
         Microsoft DialoGPT.
         How can I help you today?

📌 Commands:
   • Type 'exit' or 'quit' to end conversation
   • Type 'history' to view conversation log
   • Type 'reset' to clear conversation context

👤 You: Hello!
🤖 Chatbot: Hi, newb here.

👤 You: I'm doing great. What is deep learning?
🤖 Chatbot: I am doing great. What is learning?

👤 You: What can I use Python for?
🤖 Chatbot: Python is one of the best programming languages, just because it's difficult for developers to learn.

👤 You: Who created Python?
🤖 Chatbot: Who designed Java? Who created Java?

👤 You: exit

🤖 Chatbot: Goodbye! Thanks for chatting with me! 👋

📊 Conversation Summary:
   Total exchanges: 4


In [14]:
# Understanding generation parameters and their effects

print("=" * 60)
print("UNDERSTANDING GENERATION PARAMETERS")
print("=" * 60)

test_input = "What is the meaning of life?"

# Parameter set 1: Conservative (Low temperature)
print("\n1️⃣ LOW TEMPERATURE (0.3) - More focused/deterministic:")
input_ids = tokenizer.encode(test_input + tokenizer.eos_token, return_tensors="pt")
output = model.generate(
    input_ids,
    max_length=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=0.3,
    top_k=50
)
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
print(f"   User: {test_input}")
print(f"   Bot:  {response}")

# Parameter set 2: Creative (High temperature)
print("\n2️⃣ HIGH TEMPERATURE (1.0) - More creative/random:")
output = model.generate(
    input_ids,
    max_length=100,
    pad_token_id=tokenizer.eos_token_id,
    do_sample=True,
    temperature=1.0,
    top_k=50
)
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
print(f"   User: {test_input}")
print(f"   Bot:  {response}")

# Parameter set 3: Beam Search
print("\n3️⃣ BEAM SEARCH (num_beams=5) - More coherent:")
output = model.generate(
    input_ids,
    max_length=100,
    pad_token_id=tokenizer.eos_token_id,
    num_beams=5,
    no_repeat_ngram_size=3
)
response = tokenizer.decode(output[:, input_ids.shape[-1]:][0], skip_special_tokens=True)
print(f"   User: {test_input}")
print(f"   Bot:  {response}")

print("\n" + "=" * 60)

UNDERSTANDING GENERATION PARAMETERS

1️⃣ LOW TEMPERATURE (0.3) - More focused/deterministic:
   User: What is the meaning of life?
   Bot:  What is the meaning of life?

2️⃣ HIGH TEMPERATURE (1.0) - More creative/random:
   User: What is the meaning of life?
   Bot:  What is life?

3️⃣ BEAM SEARCH (num_beams=5) - More coherent:
   User: What is the meaning of life?
   Bot:  What is life?



In [15]:
# Display model information
print("=" * 60)
print("MODEL ARCHITECTURE SUMMARY")
print("=" * 60)

print(f"\n📌 Model Name: {model_name}")
print(f"📌 Model Type: {model.config.model_type}")
print(f"📌 Number of Parameters: {model.num_parameters():,}")
print(f"📌 Number of Layers: {model.config.n_layer}")
print(f"📌 Hidden Size: {model.config.n_embd}")
print(f"📌 Number of Attention Heads: {model.config.n_head}")
print(f"📌 Vocabulary Size: {model.config.vocab_size}")
print(f"📌 Max Position Embeddings: {model.config.n_positions}")

print("\n" + "=" * 60)
print("PIPELINE FLOW")
print("=" * 60)
print("""
┌─────────────┐    ┌──────────────┐    ┌────────────────┐
│  User Input  │───▶│  Tokenizer   │───▶│  Token IDs     │
│  (Text)      │    │  (Encoding)  │    │  (Tensor)      │
└─────────────┘    └──────────────┘    └────────────────┘
                                              │
                                              ▼
┌─────────────┐    ┌──────────────┐    ┌────────────────┐
│   Display    │◀──│  Tokenizer   │◀──│  DialoGPT      │
│   Output     │    │  (Decoding)  │    │  Model         │
└─────────────┘    └──────────────┘    └────────────────┘
        │
        ▼
   Loop Until
   User Types
    'exit'
""")

MODEL ARCHITECTURE SUMMARY

📌 Model Name: microsoft/DialoGPT-small
📌 Model Type: gpt2
📌 Number of Parameters: 163,037,184
📌 Number of Layers: 12
📌 Hidden Size: 768
📌 Number of Attention Heads: 12
📌 Vocabulary Size: 50257
📌 Max Position Embeddings: 1024

PIPELINE FLOW

┌─────────────┐    ┌──────────────┐    ┌────────────────┐
│  User Input  │───▶│  Tokenizer   │───▶│  Token IDs     │
│  (Text)      │    │  (Encoding)  │    │  (Tensor)      │
└─────────────┘    └──────────────┘    └────────────────┘
                                              │
                                              ▼
┌─────────────┐    ┌──────────────┐    ┌────────────────┐
│   Display    │◀──│  Tokenizer   │◀──│  DialoGPT      │
│   Output     │    │  (Decoding)  │    │  Model         │
└─────────────┘    └──────────────┘    └────────────────┘
        │
        ▼
   Loop Until 
   User Types
    'exit'

